# Chapitre 4 — Architecture GMIC

**🎯 Objectif :** comprendre l'architecture **GMIC** (branche globale + branche locale sur patchs + fusion) et faire une **inférence** avec un modèle pré-entraîné NYU, sans avoir besoin des données RSNA.
**⏱ Durée :** ~1–2 min (pas d'entraînement ; images d'exemple fournies par le dépôt GMIC).

**GMIC** (*Globally-aware Multiple Instance Classifier*, Shen et al. 2020) traite
une mammographie **en pleine résolution** sans la compresser. Il combine :

- une **branche globale** (`ds_net`) qui voit l'image entière en basse résolution
  et produit une **carte de saillance** 46×30 ;
- une **branche locale** (`dn_resnet`) qui zoome sur **K=6 patchs** 256×256 autour
  des zones suspectes ;
- une **fusion** des deux pour le score final.

GMIC est le sous-module `modules/GMIC` (monté dans le conteneur), avec ses **5 poids
pré-entraînés NYU** (`models/sample_model_*.p`) et des **images d'exemple** — ce
notebook tourne sans données RSNA. La publication utilise la **moyenne des 5 modèles**
(un ensemble) ; ici on charge le modèle 1.

In [ ]:
import os, sys, glob, warnings
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
import cv2
from course_utils import flowchart, gmic_dir, pick_device
warnings.filterwarnings('ignore')
torch.manual_seed(0)

GMIC_DIR = gmic_dir()                 # sous-module modules/GMIC
sys.path.insert(0, GMIC_DIR)
DEVICE = pick_device(verbose=False)
print('Device :', DEVICE)

flowchart([
    'Image 2944x1920',
    'Branche globale ds_net -> saliency 46x30 -> y_global',
    'Selection gloutonne de K=6 ROI 256x256',
    'Branche locale dn_resnet sur les 6 patchs',
    'Attention MIL -> z, y_local',
    'Fusion (global + z) -> y_fusion',
], title='Pipeline GMIC')

In [ ]:
# Une image d'exemple GMIC, préparée comme au chapitre 2.5 (crop + resize 2944x1920 + z-score).
GMIC_H, GMIC_W = 2944, 1920
imgs = sorted(glob.glob(os.path.join(GMIC_DIR, 'sample_data/images', '*.png')))
assert imgs, 'Images GMIC introuvables.'
raw = cv2.imread(imgs[0], cv2.IMREAD_UNCHANGED).astype(np.float32)

mask = raw > 0.05 * raw.max()
ys, xs = np.where(mask)
raw = raw[ys.min():ys.max() + 1, xs.min():xs.max() + 1]      # crop fond noir
raw = cv2.resize(raw, (GMIC_W, GMIC_H), interpolation=cv2.INTER_AREA)
norm = (raw - raw.mean()) / max(raw.std(), 1e-5)            # z-score
img_t = torch.tensor(norm[None, None], dtype=torch.float32).to(DEVICE)
print('Entrée modèle :', tuple(img_t.shape), '(N, C, H, W)')
print('Image :', os.path.basename(imgs[0]))

## Partie I — La brique de base : `BasicBlock`

GMIC est bâti sur des blocs résiduels. `BasicBlockV1` (branche locale) empile deux
`conv3×3 → BN → ReLU` et ajoute une **connexion résiduelle** : la sortie est
`F(x) + x`. Ce raccourci stabilise les gradients et permet d'entraîner des réseaux
profonds. Visualisons ses feature maps sur un patch.

In [ ]:
from src.modeling.modules import BasicBlockV1

block = BasicBlockV1(inplanes=64, planes=64)
print('BasicBlockV1(64->64) :', sum(p.numel() for p in block.parameters()), 'params')

crop = norm[GMIC_H // 2 - 128: GMIC_H // 2 + 128, GMIC_W // 2 - 128: GMIC_W // 2 + 128]
x = torch.tensor(crop[None, None], dtype=torch.float32)
# stem = simple adaptateur 1->64 canaux (poids ALÉATOIRES, non entraînés) : BasicBlockV1
# attend 64 canaux en entrée, le patch n'en a qu'1. Sert juste à observer la FORME des
# features produites par le bloc résiduel, pas des activations GMIC réelles.
stem = nn.Conv2d(1, 64, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    feats = block(stem(x))            # (1, 64, 256, 256)

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(crop, cmap='gray'); axes[0].set_title('Patch 256x256'); axes[0].axis('off')
for i, ax in enumerate(axes[1:]):
    ax.imshow(feats[0, i].numpy(), cmap='viridis'); ax.set_title(f'feature {i+1}'); ax.axis('off')
plt.suptitle('BasicBlockV1 — 4 premiers filtres'); plt.tight_layout(); plt.show()

## Partie II — Le modèle complet + poids NYU

On instancie le `GMIC` officiel et on charge `sample_model_1.p` en **`strict=False`**
(le checkpoint contient quelques clés non utilisées au forward — on les ignore).
`percent_t=0.02` : le score global agrège les **2 %** de cellules les plus actives.
`cam_size=(46, 30)` = 2944/64 × 1920/64.

In [ ]:
from src.modeling.gmic import GMIC

params = {
    'device_type': 'gpu' if DEVICE.type == 'cuda' else 'cpu',
    'gpu_number': 0,
    'cam_size': (46, 30),
    'K': 6,
    'crop_shape': (256, 256),
    'percent_t': 0.02,
    'use_v1_global': False,
    'post_processing_dim': 256,
    'num_classes': 2,
}
model = GMIC(params).to(DEVICE).eval()
# weights_only=True : vérifié -> ce checkpoint NYU est un simple state_dict de tenseurs
# (torch.load le charge sans exécuter de code arbitraire, la voie sûre par défaut).
ckpt = torch.load(os.path.join(GMIC_DIR, 'models/sample_model_1.p'),
                  map_location=DEVICE, weights_only=True)
missing, unexpected = model.load_state_dict(ckpt, strict=False)
print('params totaux :', f'{sum(p.numel() for p in model.parameters()):,}')
print('clés manquantes :', len(missing), '| clés en trop :', len(unexpected))

with torch.no_grad():
    y_fusion = model(img_t)           # déclenche tout le pipeline
print('y_fusion :', y_fusion.detach().cpu().numpy().round(4))

Après le forward, GMIC stocke chaque étape intermédiaire en attribut :
`saliency_map`, `patch_locations`, `patches`, `patch_attns`, `y_global`,
`y_local`, `y_fusion`. On les visualise une à une.

### Étape 1 — Carte de saillance (branche globale)

La branche globale produit une carte 46×30 : la probabilité de malignité par
cellule. Le score `y_global` est la moyenne des 2 % de cellules les plus actives.

In [ ]:
sal = model.saliency_map.detach().cpu().numpy()[0]        # (num_classes, 46, 30)
sal_malig = sal[-1]                                        # dernier canal = malignant
thr = np.percentile(sal_malig, 98)

fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(raw, cmap='gray', aspect='auto'); ax[0].set_title('Image'); ax[0].axis('off')
im = ax[1].imshow(sal_malig, cmap='hot'); ax[1].set_title('Saliency 46x30 P(malin)')
plt.colorbar(im, ax=ax[1], fraction=0.04)
ax[2].imshow(sal_malig, cmap='gray'); ax[2].imshow(sal_malig >= thr, cmap='Reds', alpha=0.6)
ax[2].set_title('Top 2% des cellules')
plt.tight_layout(); plt.show()

### Étape 2 — Sélection gloutonne des K=6 ROI

À partir de la saillance, GMIC choisit 6 fenêtres 256×256 **non chevauchantes**
aux scores les plus élevés (`patch_locations`, coordonnées en pixels de l'image).

In [ ]:
locs = np.asarray(model.patch_locations)[0]              # (K, 2) -> (row, col)
fig, ax = plt.subplots(figsize=(5, 8))
ax.imshow(raw, cmap='gray', aspect='auto')
colors = plt.cm.Set1(np.linspace(0, 1, len(locs)))
for k, (loc, col) in enumerate(zip(locs, colors)):
    r, c = int(loc[0]), int(loc[1])
    ax.add_patch(plt.Rectangle((c, r), 256, 256, lw=2, edgecolor=col, facecolor='none'))
    ax.text(c + 6, r + 30, f'ROI {k+1}', color=col, fontsize=9, fontweight='bold')
ax.set_title('K=6 ROI'); ax.axis('off'); plt.tight_layout(); plt.show()

### Étape 3 & 4 — Branche locale + attention

Chaque patch passe dans la branche locale (vecteur 512-dim), puis l'**attention de
Ilse et al. 2018** pondère les 6 patchs (poids α, somme = 1). Plus α est grand, plus
ce patch a pesé dans la décision.

In [ ]:
patches = np.asarray(model.patches)[0]                   # (K, 256, 256)
attn = np.asarray(model.patch_attns).reshape(-1)         # (K,)
best = int(attn.argmax())
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for k, ax in enumerate(axes.flat):
    ax.imshow(patches[k], cmap='gray')
    ax.set_title(f'ROI {k+1}  alpha={attn[k]:.3f}',
                 fontweight='bold' if k == best else 'normal')
    ax.axis('off')
plt.suptitle('Patchs locaux + poids d attention'); plt.tight_layout(); plt.show()

### Étape 5 — Fusion et scores finaux

Trois scores de malignité : `y_global` (branche globale seule), `y_local`
(branche locale + attention) et `y_fusion` (la combinaison = sortie du modèle).

In [ ]:
def malig(t):
    return float(t.detach().cpu().reshape(-1)[-1])       # dernier canal = malignant

print(f'y_global  (branche globale)        : {malig(model.y_global):.4f}')
print(f'y_local   (branche locale + attn)  : {malig(model.y_local):.4f}')
print(f'y_fusion  (FUSION = sortie modèle) : {malig(model.y_fusion):.4f}')

## Récap

GMIC enchaîne : **saillance globale** → **sélection de ROI** → **zoom local** →
**attention** → **fusion**. Ce qui le distingue d'un simple ResNet sur
mammographies : la **pleine résolution** (pas de compression qui efface les
microcalcifications) et les **poids NYU** (entraînés sur ~230 000 mammographies).

> **Ensemble** : rejouer ce notebook avec `sample_model_2.p` … `_5.p` (chacun a son
> propre `percent_t`) et moyenner les `y_fusion` reproduit l'ensemble du papier.

Chapitre suivant → `05_gmic_finetuning.ipynb` : adapter ces poids à RSNA.

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide la brique de base et la logique top-T% SANS poids ni image réelle.
import sys, numpy as np, torch
from course_utils import gmic_dir
sys.path.insert(0, gmic_dir())

# 1. BasicBlockV1 : conserve la forme spatiale (résiduel 64->64)
try:
    from src.modeling.modules import BasicBlockV1
    _blk = BasicBlockV1(inplanes=64, planes=64).eval()
    with torch.no_grad():
        _y = _blk(torch.randn(1, 64, 32, 32))
    assert tuple(_y.shape) == (1, 64, 32, 32), _y.shape
    _blk_ok = True
except ImportError:
    _blk_ok = False        # repo GMIC absent (hors conteneur) -> partie sautée

# 2. Sélection top-2% sur une saliency synthétique (pur numpy)
_sal = np.random.rand(46, 30)
_frac = (_sal >= np.percentile(_sal, 98)).mean()
assert 0.015 < _frac < 0.05, f'top-2% incohérent : {_frac:.3f}'
print(f'✅ top-2% OK ({_frac*100:.1f}% des cellules)'
      + (' | BasicBlockV1 forward OK' if _blk_ok else ' | (BasicBlockV1 sauté : repo GMIC absent)'))